# Aula 21 — Laboratório: clustering como hipótese geométrica

Este laboratório compara **K-Means, clustering hierárquico e DBSCAN** sem usar rótulos durante o ajuste. Como os dados são sintéticos, conhecemos a estrutura geradora apenas para diagnosticar falhas com ARI; em um projeto real, essa referência normalmente não existe.

**Protocolo:** dados gerados localmente, seed fixa, uma decisão por experimento, métricas internas separadas de referência externa, estabilidade por perturbação e verificações automáticas. Os gráficos são saídas didáticas; não são prova isolada de clusters.

## Dependências e reprodutibilidade

Requer Python ≥ 3.10, NumPy ≥ 1.26, Matplotlib ≥ 3.8, SciPy ≥ 1.11 e scikit-learn ≥ 1.4. O notebook não baixa dados, não usa credenciais e força warnings a virar erros.

In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scipy
import sklearn
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("error")
SEED = 20260908
rng = np.random.default_rng(SEED)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("seed:", SEED)

## 1. Lloyd com NumPy

Implementamos atribuição e atualização para observar a função objetivo. O exemplo tem três grupos compactos e centroides iniciais deliberadamente imperfeitos.

In [ ]:
def inertia(X, labels, centers):
    return float(np.sum((X - centers[labels]) ** 2))


def lloyd(X, initial_centers, max_iter=30):
    centers = np.asarray(initial_centers, dtype=float).copy()
    history = []
    for _ in range(max_iter):
        squared_distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = squared_distances.argmin(axis=1)
        value = inertia(X, labels, centers)
        history.append(value)
        new_centers = np.vstack([X[labels == k].mean(axis=0) for k in range(len(centers))])
        if np.allclose(new_centers, centers):
            centers = new_centers
            history.append(inertia(X, labels, centers))
            break
        centers = new_centers
    final_distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    labels = final_distances.argmin(axis=1)
    final_value = inertia(X, labels, centers)
    if not np.isclose(history[-1], final_value):
        history.append(final_value)
    return centers, labels, np.asarray(history)


X_small = np.array([[0., 0.], [0., 1.], [1., 0.], [8., 8.], [8., 9.], [9., 8.], [4., 9.], [5., 9.], [4.5, 10.]])
initial = np.array([[0., 0.], [5., 5.], [9., 9.]])
centers_np, labels_np, objective_history = lloyd(X_small, initial)
print("histórico da inertia:", np.round(objective_history, 6).tolist())
print("centroides finais:\n", np.round(centers_np, 6))

In [ ]:
reference = KMeans(n_clusters=3, init=initial, n_init=1, random_state=SEED).fit(X_small)
print("inertia NumPy:", round(objective_history[-1], 6))
print("inertia scikit-learn:", round(reference.inertia_, 6))
print("ARI entre implementações:", adjusted_rand_score(labels_np, reference.labels_))

**Leitura:** a inertia cai a cada alternância, mas isso garante apenas convergência local. Rótulos numéricos podem ser permutados; por isso comparamos partições usando ARI.

## 2. A unidade muda a resposta

O primeiro eixo contém dois grupos; o segundo é ruído. Multiplicamos somente o ruído por 100, simulando troca de unidade. Comparamos K-Means bruto e um ajuste após `StandardScaler`. O rótulo gerador é usado apenas na auditoria posterior.

In [ ]:
n_scale = 800
y_scale = np.repeat([0, 1], n_scale // 2)
x_signal = rng.normal(loc=np.where(y_scale == 0, -3.0, 3.0), scale=0.7)
x_noise = rng.normal(size=n_scale)
X_units = np.column_stack([x_signal, 100.0 * x_noise])

raw_labels = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit_predict(X_units)
X_units_std = StandardScaler().fit_transform(X_units)
scaled_labels = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit_predict(X_units_std)
ari_raw = adjusted_rand_score(y_scale, raw_labels)
ari_scaled = adjusted_rand_score(y_scale, scaled_labels)
print("ARI sem escala:", round(ari_raw, 6))
print("ARI padronizado:", round(ari_scaled, 6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
axes[0].scatter(X_units_std[:, 0], X_units_std[:, 1], c=raw_labels, s=9, cmap="viridis")
axes[0].set_title("K-Means nos valores brutos")
axes[1].scatter(X_units_std[:, 0], X_units_std[:, 1], c=scaled_labels, s=9, cmap="viridis")
axes[1].set_title("K-Means após padronização")
for ax in axes:
    ax.set(xlabel="sinal padronizado", ylabel="ruído padronizado")
plt.tight_layout()
plt.show()

A padronização recupera a separação neste processo conhecido. Em dados reais ela não é regra automática: escolher pesos iguais para desvios-padrão também expressa uma hipótese.

## 3. Duas luas: três hipóteses geométricas

Geramos duas formas curvas, padronizamos sem olhar os rótulos e ajustamos três modelos. Para DBSCAN, silhouette é calculada apenas quando restam pelo menos dois clusters; reportamos também a fração excluída como ruído.

In [ ]:
X_moons, truth_moons = make_moons(n_samples=900, noise=0.065, random_state=SEED)
X_moons = StandardScaler().fit_transform(X_moons)

models = {
    "K-Means": KMeans(n_clusters=2, n_init=30, random_state=SEED),
    "Ward": AgglomerativeClustering(n_clusters=2, linkage="ward"),
    "DBSCAN": DBSCAN(eps=0.23, min_samples=8),
}

moon_results = {}
moon_labels = {}
for name, model in models.items():
    labels = model.fit_predict(X_moons)
    moon_labels[name] = labels
    keep = labels != -1
    clusters = len(set(labels[keep]))
    noise_fraction = float((~keep).mean())
    sil = silhouette_score(X_moons[keep], labels[keep]) if clusters >= 2 else np.nan
    ari = adjusted_rand_score(truth_moons, labels)
    moon_results[name] = (clusters, noise_fraction, sil, ari)
    print(f"{name:8s} clusters={clusters} ruído={noise_fraction:.4f} silhouette={sil:.6f} ARI={ari:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), sharex=True, sharey=True)
for ax, (name, labels) in zip(axes, moon_labels.items()):
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, s=8, cmap="viridis")
    ax.set_title(name)
    ax.set(xlabel="feature 1", ylabel="feature 2")
plt.tight_layout()
plt.show()

**Contraprova:** uma silhouette interna pode preferir a partição convexa de K-Means mesmo quando ela contradiz as duas luas geradoras. ARI está disponível aqui porque controlamos a simulação; em produção, estabilidade e validação de domínio ganham importância.

## 4. Hierarquia e linkage

O dendrograma registra todas as fusões para uma subamostra. Depois comparamos `single`, `complete`, `average` e `ward` no conjunto inteiro. O cálculo não consulta `truth_moons`; ela aparece apenas na avaliação didática.

In [ ]:
sample_idx = rng.choice(len(X_moons), size=100, replace=False)
Z = linkage(X_moons[sample_idx], method="ward", metric="euclidean")
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z, no_labels=True, color_threshold=None, ax=ax)
ax.set(title="Dendrograma Ward — subamostra", xlabel="observações", ylabel="distância de fusão")
plt.tight_layout()
plt.show()

linkage_ari = {}
for method in ["single", "complete", "average", "ward"]:
    labels = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(X_moons)
    linkage_ari[method] = adjusted_rand_score(truth_moons, labels)
print("ARI por linkage:", {k: round(v, 6) for k, v in linkage_ari.items()})

## 5. Estabilidade sob perturbações

Adicionamos ruído pequeno às mesmas observações e refazemos cada método 20 vezes. Como os pontos correspondem um a um, ARI mede concordância com a partição-base sem exigir alinhamento dos números dos labels.

In [ ]:
stability_rng = np.random.default_rng(SEED + 1)
stability = {name: [] for name in models}
for _ in range(20):
    X_perturbed = X_moons + stability_rng.normal(0, 0.015, size=X_moons.shape)
    for name, model in models.items():
        perturbed_labels = model.fit_predict(X_perturbed)
        stability[name].append(adjusted_rand_score(moon_labels[name], perturbed_labels))

for name, scores in stability.items():
    print(f"{name:8s} ARI estabilidade={np.mean(scores):.6f} ± {np.std(scores):.6f}; mínimo={np.min(scores):.6f}")

Estabilidade depende da perturbação escolhida. Ruído de 0,015 representa uma hipótese sobre erro de medição; outra aplicação exige outro mecanismo, além de reamostragem por entidade quando houver dependência.

## 6. Densidades diferentes: limite de um único `eps`

Criamos um grupo compacto e outro disperso próximos. Varremos `eps` sem escolher o melhor pelo rótulo. A tabela mostra o compromisso entre fragmentar a região esparsa, classificar muitos pontos como ruído e fundir regiões.

In [ ]:
X_dense, y_dense = make_blobs(
    n_samples=[350, 350],
    centers=[(0.0, 0.0), (1.2, 0.0)],
    cluster_std=[0.10, 0.55],
    random_state=SEED,
)
X_dense = StandardScaler().fit_transform(X_dense)

density_results = []
for eps in [0.08, 0.12, 0.18, 0.28, 0.42]:
    labels = DBSCAN(eps=eps, min_samples=8).fit_predict(X_dense)
    keep = labels != -1
    clusters = len(set(labels[keep]))
    noise = float((~keep).mean())
    ari = adjusted_rand_score(y_dense, labels)
    density_results.append((eps, clusters, noise, ari))
    print(f"eps={eps:.2f} clusters={clusters:2d} ruído={noise:.4f} ARI didático={ari:.6f}")

In [ ]:
chosen_eps = [0.08, 0.18, 0.42]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), sharex=True, sharey=True)
for ax, eps in zip(axes, chosen_eps):
    labels = DBSCAN(eps=eps, min_samples=8).fit_predict(X_dense)
    ax.scatter(X_dense[:, 0], X_dense[:, 1], c=labels, s=8, cmap="viridis")
    ax.set_title(f"eps={eps}")
    ax.set(xlabel="feature 1", ylabel="feature 2")
plt.tight_layout()
plt.show()

## 7. Verificações automáticas

Os asserts verificam monotonicidade da função objetivo, equivalência com a biblioteca, efeito de escala, falha convexa nas luas, desempenho de DBSCAN, invariância de labels via ARI e os padrões de estabilidade observados. Limiares são específicos desta seed e destes dados sintéticos.

In [ ]:
assert np.all(np.diff(objective_history) <= 1e-10)
assert np.isclose(objective_history[-1], reference.inertia_, atol=1e-10)
assert adjusted_rand_score(labels_np, reference.labels_) == 1.0
assert ari_raw < 0.10
assert ari_scaled > 0.90
assert moon_results["K-Means"][3] < 0.60
assert moon_results["DBSCAN"][3] > 0.95
assert moon_results["DBSCAN"][1] < 0.05
assert linkage_ari["single"] > linkage_ari["ward"]
assert np.mean(stability["K-Means"]) > 0.98
assert np.mean(stability["DBSCAN"]) > 0.98
assert np.mean(stability["Ward"]) < 0.80
assert adjusted_rand_score([0, 0, 1, 1], [1, 1, 0, 0]) == 1.0
assert max(row[2] for row in density_results) > 0.20
assert density_results[-1][1] == 1

print("Todas as verificações passaram.")
print({
    "inertia_final": round(float(objective_history[-1]), 6),
    "ari_sem_escala": round(float(ari_raw), 6),
    "ari_padronizado": round(float(ari_scaled), 6),
    "ari_kmeans_luas": round(float(moon_results["K-Means"][3]), 6),
    "silhouette_kmeans_luas": round(float(moon_results["K-Means"][2]), 6),
    "ari_dbscan_luas": round(float(moon_results["DBSCAN"][3]), 6),
    "silhouette_dbscan_luas": round(float(moon_results["DBSCAN"][2]), 6),
    "estabilidade_kmeans": round(float(np.mean(stability["K-Means"])), 6),
    "estabilidade_ward": round(float(np.mean(stability["Ward"])), 6),
    "estabilidade_dbscan": round(float(np.mean(stability["DBSCAN"])), 6),
})

## Conclusões

- Lloyd reduziu a inertia monotonamente e coincidiu com a implementação de referência no exemplo.
- Trocar a unidade do ruído redefiniu a proximidade; padronização recuperou a estrutura somente porque conhecíamos a simulação.
- K-Means e Ward impuseram cortes convexos às luas; DBSCAN acompanhou sua conectividade por densidade.
- Silhouette e ARI responderam perguntas distintas e chegaram a favorecer narrativas diferentes.
- Linkage alterou a hierarquia e a partição final.
- Um `eps` global produziu compromissos em densidades diferentes.
- Estabilidade apoiou a análise, mas não forneceu significado de domínio nem justificativa para decisões.

Em aplicação real, registre representação, métrica, parâmetros, cobertura, estabilidade, política para novos pontos e impacto. Não transforme a saída exploratória em identidade ou causalidade.

## Próxima aula

Na [Aula 22](../aulas/22-reducao-dimensionalidade-ml.md), examinaremos PCA, t-SNE e UMAP. Mapas 2D serão tratados como transformações com objetivos e distorções próprios, não como validação automática dos clusters desta aula.